In [1]:
import numpy as np
import pandas as pd
from scapy.all import rdpcap, raw
import matplotlib.pyplot as plt
import seaborn as sns



In [2]:
%whos


Variable   Type        Data/Info
--------------------------------
np         module      <module 'numpy' from '/ho<...>kages/numpy/__init__.py'>
pd         module      <module 'pandas' from '/h<...>ages/pandas/__init__.py'>
plt        module      <module 'matplotlib.pyplo<...>es/matplotlib/pyplot.py'>
raw        function    <function raw at 0x7d4e1f6b5620>
rdpcap     function    <function rdpcap at 0x7d4e1daf7380>
sns        module      <module 'seaborn' from '/<...>ges/seaborn/__init__.py'>


In [3]:
!free -h


               total        used        free      shared  buff/cache   available
Mem:            31Gi       6.1Gi        19Gi       592Mi       6.5Gi        24Gi
Swap:           14Gi       2.4Gi        12Gi


In [4]:
# --- KONFIGURASI DAN PATH FILE ---
BASE_PATH = '/home/dani/Documents/tugas akhir/data/TOW-IDS/TOW-IDS-20250906T180910Z-1-001/TOW-IDS/Automotive Ethernet Dataset/'
TRAIN_PCAP_PATH = BASE_PATH + 'Automotive_Ethernet_with_Attack_original_10_17_19_50_training.pcap'
TRAIN_LABELS_PATH = BASE_PATH + 'y_train.csv'

# Kita definisikan juga path untuk data testing untuk nanti
TEST_PCAP_PATH = BASE_PATH + 'Automotive_Ethernet_with_Attack_original_10_17_20_04_test.pcap'
TEST_LABELS_PATH = BASE_PATH + 'y_test.csv'

print("Langkah 1 Selesai: Library dan path sudah siap.")

Langkah 1 Selesai: Library dan path sudah siap.


In [5]:
# --- MEMUAT LABEL TRAINING ---
print("Memuat file label dari:", TRAIN_LABELS_PATH)
df_labels_train = pd.read_csv(TRAIN_LABELS_PATH , header=None)

# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_train1 = df_labels_train.iloc[:, 1].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_train1)} label.")
print("Contoh 5 label pertama:", labels_train1[:5])
print("Distribusi Label:")
print(df_labels_train.iloc[:, 1].value_counts())

Memuat file label dari: /home/dani/Documents/tugas akhir/data/TOW-IDS/TOW-IDS-20250906T180910Z-1-001/TOW-IDS/Automotive Ethernet Dataset/y_train.csv

Berhasil memuat 1203737 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
1
Normal      954912
Abnormal    248825
Name: count, dtype: int64


In [6]:
# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_train2 = df_labels_train.iloc[:, 2].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_train2)} label.")
print("Contoh 5 label pertama:", labels_train2[:5])
print("Distribusi Label:")
print(df_labels_train.iloc[:, 2].value_counts())


Berhasil memuat 1203737 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
2
Normal    954912
C_D        85466
P_I        64635
F_I        35112
M_F        33765
C_R        29847
Name: count, dtype: int64


In [8]:

def extract_packets(pcap_path, n_bytes=64):
    packets = rdpcap(pcap_path)
    data = []

    for pkt in packets:
        pkt_bytes = raw(pkt)

        if len(pkt_bytes) < n_bytes:
            pkt_bytes = pkt_bytes + bytes(n_bytes - len(pkt_bytes))  # zero padding
        else:
            pkt_bytes = pkt_bytes[:n_bytes]  # truncate

        data.append(np.frombuffer(pkt_bytes, dtype=np.uint8))

    return np.array(data)  # (num_packets, 64)


In [9]:

def build_sequences(X_pkt, y_pkt, window=64, step=64):
    X_seq, y_seq = [], []

    for i in range(0, len(X_pkt) - window + 1, step):
        window_data = X_pkt[i:i+window]
        window_label = y_pkt[i:i+window]

        X_seq.append(window_data.T)  # (64, 64) → (n × l)
        y_seq.append(1 if np.any(window_label == 1) else 0)

    return np.array(X_seq), np.array(y_seq)


In [10]:
def build_sequences_multiclass(X_pkt, y_pkt, window=64, step=64):
    X_seq, y_seq = [], []

    for i in range(0, len(X_pkt) - window + 1, step):
        window_data = X_pkt[i:i+window]
        window_label = y_pkt[i:i+window]

        X_seq.append(window_data.T)

        # buang normal kalau ada attack
        attack_only = window_label[window_label > 0]

        if len(attack_only) > 0:
            vals, cnts = np.unique(attack_only, return_counts=True)
            y_seq.append(vals[np.argmax(cnts)])
        else:
            y_seq.append(0)

    return np.array(X_seq), np.array(y_seq)


training binary class

In [11]:
x_pkt_train=extract_packets(TRAIN_PCAP_PATH)

In [12]:
binary_labels_text   = df_labels_train.iloc[:, 1]
y_label_train  = binary_labels_text.map({'Normal': 0, 'Abnormal': 1}).values

In [13]:

x_seq_train, y_seq_train = build_sequences(x_pkt_train,y_label_train)

In [14]:
x_seq_train.shape, y_seq_train.shape

((18808, 64, 64), (18808,))

training multiclass

In [16]:
multi_labels_text = df_labels_train.iloc[:, 2]
y_label_trainmulti  = multi_labels_text.map({'Normal': 0, 'C_D': 1, 'P_I': 2, 'F_I': 3, 'M_F': 4, 'C_R': 5}).values



In [18]:
x_seq_trainmulti, y_seq_trainmulti = build_sequences_multiclass(x_pkt_train,y_label_trainmulti)

In [19]:
x_seq_trainmulti.shape, y_seq_trainmulti.shape

((18808, 64, 64), (18808,))

In [20]:
np.unique(x_seq_trainmulti)

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

In [21]:
np.unique(y_seq_trainmulti)

array([0, 1, 2, 3, 4, 5])

In [22]:
np.unique(y_seq_train)

array([0, 1])

test binary class

In [24]:
# --- MEMUAT LABEL Testing ---
print("Memuat file label dari:", TEST_LABELS_PATH)
df_labels_test = pd.read_csv(TEST_LABELS_PATH , header=None)

# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_test1 = df_labels_test.iloc[:, 1].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_test1)} label.")
print("Contoh 5 label pertama:", labels_test1[:5])
print("Distribusi Label:")
print(df_labels_test.iloc[:, 1].value_counts())
print(df_labels_test.head())


Memuat file label dari: /home/dani/Documents/tugas akhir/data/TOW-IDS/TOW-IDS-20250906T180910Z-1-001/TOW-IDS/Automotive Ethernet Dataset/y_test.csv

Berhasil memuat 791611 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
1
Normal      660777
Abnormal    130834
Name: count, dtype: int64
   0       1       2
0  1  Normal  Normal
1  2  Normal  Normal
2  3  Normal  Normal
3  4  Normal  Normal
4  5  Normal  Normal


In [25]:
# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_test2 = df_labels_test.iloc[:, 2].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_test2)} label.")
print("Contoh 5 label pertama:", labels_test2[:5])
print("Distribusi Label:")
print(df_labels_test.iloc[:, 2].value_counts())


Berhasil memuat 791611 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
2
Normal    660777
C_D        41203
C_R        29847
P_I        26013
F_I        16962
M_F        16809
Name: count, dtype: int64


In [26]:
x_pkt_test=extract_packets(TEST_PCAP_PATH)

In [27]:
binary_labels_text   = df_labels_test.iloc[:, 1]
y_label_test = binary_labels_text.map({'Normal': 0, 'Abnormal': 1}).values

In [28]:

x_seq_test, y_seq_test = build_sequences(x_pkt_test, y_label_test)
x_seq_test.shape, y_seq_test.shape

((12368, 64, 64), (12368,))

In [23]:
np.unique(x_seq_test)

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

In [29]:
np.unique(y_seq_test)

array([0, 1])

multiclass

In [30]:
multi_labels_test = df_labels_test.iloc[:, 2]
y_label_test  = multi_labels_test.map({'Normal': 0, 'C_D': 1, 'P_I': 2, 'F_I': 3, 'M_F': 4, 'C_R': 5}).values


In [31]:
y_label_testmulti = multi_labels_test.map({'Normal': 0, 'C_D': 1, 'P_I': 2, 'F_I': 3, 'M_F': 4, 'C_R': 5}).values
np.unique(y_label_testmulti)

label_map = {
    0: 'Normal',
    1: 'C_D',
    2: 'P_I',
    3: 'F_I',
    4: 'M_F',
    5: 'C_R'
}

labels, counts = np.unique(y_label_testmulti, return_counts=True)

for l, c in zip(labels, counts):
    print(f"{label_map[l]} ({l}): {c}")


Normal (0): 660777
C_D (1): 41203
P_I (2): 26013
F_I (3): 16962
M_F (4): 16809
C_R (5): 29847


In [32]:
x_seq_testmulti, y_seq_testmulti = build_sequences_multiclass(x_pkt_test, y_label_testmulti)
x_seq_testmulti.shape, y_seq_testmulti.shape

((12368, 64, 64), (12368,))

In [27]:
np.unique(x_seq_testmulti)

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

In [33]:
np.unique(y_seq_testmulti)
labels, counts = np.unique(y_seq_testmulti, return_counts=True)

for l, c in zip(labels, counts):
    print(f"Label {l}: {c}")


Label 0: 6846
Label 1: 1356
Label 2: 1192
Label 3: 570
Label 4: 793
Label 5: 1611


In [34]:
np.unique(y_seq_testmulti)

array([0, 1, 2, 3, 4, 5])

normalize

In [35]:
import numpy as np
import gc
def normalize_minmax(X):
    return X.astype(np.float32) / 255.0

In [36]:
x_seq_train = normalize_minmax(x_seq_train)
x_seq_trainmulti = normalize_minmax(x_seq_trainmulti)

In [37]:
np.unique(x_seq_train)

array([0.        , 0.00392157, 0.00784314, 0.01176471, 0.01568628,
       0.01960784, 0.02352941, 0.02745098, 0.03137255, 0.03529412,
       0.03921569, 0.04313726, 0.04705882, 0.05098039, 0.05490196,
       0.05882353, 0.0627451 , 0.06666667, 0.07058824, 0.07450981,
       0.07843138, 0.08235294, 0.08627451, 0.09019608, 0.09411765,
       0.09803922, 0.10196079, 0.10588235, 0.10980392, 0.11372549,
       0.11764706, 0.12156863, 0.1254902 , 0.12941177, 0.13333334,
       0.13725491, 0.14117648, 0.14509805, 0.14901961, 0.15294118,
       0.15686275, 0.16078432, 0.16470589, 0.16862746, 0.17254902,
       0.1764706 , 0.18039216, 0.18431373, 0.1882353 , 0.19215687,
       0.19607843, 0.2       , 0.20392157, 0.20784314, 0.21176471,
       0.21568628, 0.21960784, 0.22352941, 0.22745098, 0.23137255,
       0.23529412, 0.23921569, 0.24313726, 0.24705882, 0.2509804 ,
       0.25490198, 0.25882354, 0.2627451 , 0.26666668, 0.27058825,
       0.27450982, 0.2784314 , 0.28235295, 0.28627452, 0.29019

In [38]:
np.unique(x_seq_trainmulti)

array([0.        , 0.00392157, 0.00784314, 0.01176471, 0.01568628,
       0.01960784, 0.02352941, 0.02745098, 0.03137255, 0.03529412,
       0.03921569, 0.04313726, 0.04705882, 0.05098039, 0.05490196,
       0.05882353, 0.0627451 , 0.06666667, 0.07058824, 0.07450981,
       0.07843138, 0.08235294, 0.08627451, 0.09019608, 0.09411765,
       0.09803922, 0.10196079, 0.10588235, 0.10980392, 0.11372549,
       0.11764706, 0.12156863, 0.1254902 , 0.12941177, 0.13333334,
       0.13725491, 0.14117648, 0.14509805, 0.14901961, 0.15294118,
       0.15686275, 0.16078432, 0.16470589, 0.16862746, 0.17254902,
       0.1764706 , 0.18039216, 0.18431373, 0.1882353 , 0.19215687,
       0.19607843, 0.2       , 0.20392157, 0.20784314, 0.21176471,
       0.21568628, 0.21960784, 0.22352941, 0.22745098, 0.23137255,
       0.23529412, 0.23921569, 0.24313726, 0.24705882, 0.2509804 ,
       0.25490198, 0.25882354, 0.2627451 , 0.26666668, 0.27058825,
       0.27450982, 0.2784314 , 0.28235295, 0.28627452, 0.29019

In [39]:
x_seq_test = normalize_minmax(x_seq_test)
x_seq_testmulti = normalize_minmax(x_seq_testmulti)

In [40]:
np.unique(x_seq_test)

array([0.        , 0.00392157, 0.00784314, 0.01176471, 0.01568628,
       0.01960784, 0.02352941, 0.02745098, 0.03137255, 0.03529412,
       0.03921569, 0.04313726, 0.04705882, 0.05098039, 0.05490196,
       0.05882353, 0.0627451 , 0.06666667, 0.07058824, 0.07450981,
       0.07843138, 0.08235294, 0.08627451, 0.09019608, 0.09411765,
       0.09803922, 0.10196079, 0.10588235, 0.10980392, 0.11372549,
       0.11764706, 0.12156863, 0.1254902 , 0.12941177, 0.13333334,
       0.13725491, 0.14117648, 0.14509805, 0.14901961, 0.15294118,
       0.15686275, 0.16078432, 0.16470589, 0.16862746, 0.17254902,
       0.1764706 , 0.18039216, 0.18431373, 0.1882353 , 0.19215687,
       0.19607843, 0.2       , 0.20392157, 0.20784314, 0.21176471,
       0.21568628, 0.21960784, 0.22352941, 0.22745098, 0.23137255,
       0.23529412, 0.23921569, 0.24313726, 0.24705882, 0.2509804 ,
       0.25490198, 0.25882354, 0.2627451 , 0.26666668, 0.27058825,
       0.27450982, 0.2784314 , 0.28235295, 0.28627452, 0.29019

split validation 30% binary

In [36]:
np.unique(x_seq_testmulti)

array([0.        , 0.00392157, 0.00784314, 0.01176471, 0.01568627,
       0.01960784, 0.02352941, 0.02745098, 0.03137255, 0.03529412,
       0.03921569, 0.04313725, 0.04705882, 0.05098039, 0.05490196,
       0.05882353, 0.0627451 , 0.06666667, 0.07058824, 0.0745098 ,
       0.07843137, 0.08235294, 0.08627451, 0.09019608, 0.09411765,
       0.09803922, 0.10196078, 0.10588235, 0.10980392, 0.11372549,
       0.11764706, 0.12156863, 0.1254902 , 0.12941176, 0.13333333,
       0.1372549 , 0.14117647, 0.14509804, 0.14901961, 0.15294118,
       0.15686275, 0.16078431, 0.16470588, 0.16862745, 0.17254902,
       0.17647059, 0.18039216, 0.18431373, 0.18823529, 0.19215686,
       0.19607843, 0.2       , 0.20392157, 0.20784314, 0.21176471,
       0.21568627, 0.21960784, 0.22352941, 0.22745098, 0.23137255,
       0.23529412, 0.23921569, 0.24313725, 0.24705882, 0.25098039,
       0.25490196, 0.25882353, 0.2627451 , 0.26666667, 0.27058824,
       0.2745098 , 0.27843137, 0.28235294, 0.28627451, 0.29019

In [41]:
from sklearn.model_selection import train_test_split

x_trainbinary, x_valbinary, y_trainbinary, y_valbinary = train_test_split(
    x_seq_train, y_seq_train,
    test_size=0.3,
    stratify=y_seq_train,
    random_state=42
)


In [42]:
from sklearn.model_selection import train_test_split

x_trainmulti, x_valmulti, y_trainmulti, y_valmulti = train_test_split(
    x_seq_trainmulti, y_seq_trainmulti,
    test_size=0.3,
    stratify=y_seq_trainmulti,
    random_state=42
)


In [43]:
np.unique(y_seq_trainmulti)

array([0, 1, 2, 3, 4, 5])

In [44]:
np.unique(y_valmulti)

array([0, 1, 2, 3, 4, 5])

In [45]:
np.unique(y_valbinary)

array([0, 1])

In [46]:
x_trainbinary.shape, y_trainbinary.shape

((13165, 64, 64), (13165,))

In [48]:
x_valbinary.shape, y_valbinary.shape

((5643, 64, 64), (5643,))

In [49]:
x_trainmulti.shape, y_trainmulti.shape

((13165, 64, 64), (13165,))

In [50]:
x_valmulti.shape, y_valmulti.shape

((5643, 64, 64), (5643,))

save comp

In [51]:
np.savez_compressed(
    "preprocessingW64S64binaryfloat32/tow_ids_binary_preprocessedW64S64.npz",
    x_train=x_trainbinary,
    y_train=y_trainbinary,
    x_val=x_valbinary,
    y_val=y_valbinary,
    x_test=x_seq_test,
    y_test=y_seq_test
)


In [52]:
np.savez_compressed(
    "preprocessingW64S64multifloat32/tow_ids_multi_preprocessedW64S64.npz",
    x_train=x_trainmulti,
    y_train=y_trainmulti,
    x_val=x_valmulti,
    y_val=y_valmulti,
    x_test=x_seq_testmulti,
    y_test=y_seq_testmulti
)


In [53]:
np.unique(y_seq_testmulti)

array([0, 1, 2, 3, 4, 5])

In [54]:
sample_1d = x_seq_test[0]

print("Panjang vektor:", len(sample_1d))
print("Contoh nilai byte:")
print(sample_1d)


Panjang vektor: 64
Contoh nilai byte:
[[0.5686275  0.5686275  0.5686275  ... 0.5686275  0.5686275  0.5686275 ]
 [0.9372549  0.9372549  0.9372549  ... 0.9372549  0.9372549  0.9372549 ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [1.         0.24313726 0.7490196  ... 0.16862746 0.6784314  0.5764706 ]
 [1.         0.10196079 0.5568628  ... 0.92941177 0.5529412  0.11764706]
 [1.         0.88235295 0.8352941  ... 0.9490196  0.95686275 0.02352941]]
